# Gender wage gap in the US, 1976-2025
Data: CPS ASEC from IPUMS. Weight: `asecwt`.

This first part connects Google Drive, loads the data, and prints the shape, the column names, and a summary table.

## 1. Libraries

In [ ]:
# Colab already has pandas, numpy, and matplotlib, so no pip install is needed
import pandas as pd  # tables
import numpy as np  # math helpers for later steps
import matplotlib.pyplot as plt  # charts for later steps
from google.colab import drive  # connect Google Drive

## 2. Settings
Change the path or the column list here.

In [ ]:
# Path to the raw data file (change here if the file moves)
data_path = "/content/drive/My Drive/jobs_assignment_data_20261104.csv"  # file in the root of My Drive

# Columns we need from the 1.6 GB file (the file has 85 columns)
needed_columns = [  # start of the list
    "year", "asecwt",  # income year and person-level survey weight
    "female", "fulltime",  # sex and full-time flag
    "age", "agegrp",  # age and age group
    "wkswork", "hrswk",  # weeks worked last year and usual hours per week
    "incwage_real", "hrwage_real",  # yearly wage and hourly wage in 2025 dollars
    "educ5", "raceeth", "marst3",  # education, race and ethnicity, marital status
    "nchild_cat", "child_u5",  # number of children, child under 5
    "occ6", "classwkr3", "metro",  # occupation, class of worker, metro area
]  # end of the list

# Show decimal numbers with 1 decimal and a comma for thousands
pd.options.display.float_format = "{:,.1f}".format  # display format for floats

## 3. Text labels for the codes
This dictionary is the one place that says what each code means. Check it against the codebook in `CLAUDE.md`.

In [ ]:
# Short description of every column we load
variable_titles = {  # start of the dictionary
    "year": "income year, 1976 to 2025",  # year
    "asecwt": "person-level survey weight (number of people in the US each row stands for)",  # weight
    "female": "sex",  # sex
    "fulltime": "full-time or part-time work last year, by usual weekly hours",  # full-time flag
    "age": "age in years, 16 to 64",  # age
    "agegrp": "age group",  # age group
    "wkswork": "weeks worked last year, 1 to 52",  # weeks worked
    "hrswk": "usual hours worked per week last year",  # usual hours
    "incwage_real": "yearly wage and salary income, in 2025 dollars",  # yearly wage
    "hrwage_real": "hourly wage, in 2025 dollars (yearly wage divided by weeks times hours)",  # hourly wage
    "educ5": "education",  # education
    "raceeth": "race and ethnicity (missing = don't know or no response)",  # race and ethnicity
    "marst3": "marital status",  # marital status
    "nchild_cat": "number of children",  # number of children
    "child_u5": "child under 5",  # child under 5
    "occ6": "occupation (missing = not in universe, military, 999, or other)",  # occupation
    "classwkr3": "class of worker (wage and salary workers only)",  # class of worker
    "metro": "metro area (missing in some years)",  # metro area
}  # end of the dictionary

# Text label for each code of each coded variable
code_labels = {  # start of the dictionary
    "female": {0: "Male", 1: "Female"},  # sex
    "fulltime": {0: "Part-time (under 35 hours)", 1: "Full-time (35+ hours)"},  # full-time flag
    "agegrp": {1: "16-24", 2: "25-34", 3: "35-44", 4: "45-54", 5: "55-64"},  # age group
    "educ5": {1: "None/preschool", 2: "Less than HS", 3: "High school", 4: "Some college", 5: "Bachelor's+"},  # education
    "raceeth": {1: "Hispanic (any race)", 2: "White (non-Hispanic)", 3: "Black (non-Hispanic)", 4: "Other"},  # race and ethnicity
    "marst3": {1: "Married", 2: "Previously married", 3: "Never married"},  # marital status
    "nchild_cat": {0: "No children", 1: "1 child", 2: "2 children", 3: "3 or more children"},  # number of children
    "child_u5": {0: "No child under 5", 1: "Child under 5"},  # child under 5
    "occ6": {1: "Managers and professionals", 2: "Technical, sales and admin", 3: "Service", 4: "Farming, forestry and fishing", 5: "Craft and repair", 6: "Operators and laborers"},  # occupation
    "classwkr3": {1: "Private", 2: "Government"},  # class of worker
    "metro": {0: "Not in a metro area", 1: "In a metro area"},  # metro area
}  # end of the dictionary

In [ ]:
print("Variable".ljust(15), "Meaning")  # header of the codebook
for name in needed_columns:  # go through each column we load
    meaning = variable_titles[name]  # short description
    if name in code_labels:  # coded variable, so spell out the codes
        code_pairs = [str(code) + " = " + label for code, label in code_labels[name].items()]  # pieces like "0 = Male"
        meaning = meaning + ": " + ", ".join(code_pairs)  # add the codes to the description
    print(name.ljust(15), meaning)  # one line per variable

## 4. Connect Google Drive

In [ ]:
drive.mount("/content/drive")  # connect Drive (click Allow when it asks)

## 5. Column names in the file

In [ ]:
file_columns = pd.read_csv(data_path, nrows=0).columns.tolist()  # read only the header row
print("Number of columns in the file:", len(file_columns))  # how many columns the file has
print("Column names:")  # label for the list
print(file_columns)  # every column name in the file
missing_names = [name for name in needed_columns if name not in file_columns]  # names we want but the file lacks
print("Needed columns not in the file:", missing_names)  # should be an empty list

## 6. Load the data and print the shape
We read only the columns we need. The raw table stays as loaded. We work on a copy.

In [ ]:
raw_data = pd.read_csv(data_path, usecols=needed_columns)  # read only the columns we need
data = raw_data.copy()  # work on a copy, the raw table stays as loaded
print("Shape of the file (rows, columns):", (len(data), len(file_columns)))  # every row is loaded, so rows match the file
print("Shape of the table we loaded (rows, columns):", data.shape)  # rows and the columns we kept

## 7. Add the text labels to the data
This adds one `_label` column for each coded variable, on our copy. The label columns are stored as categories, so they use little memory and sort in code order. The check should show 0 values with no label for every variable.

In [ ]:
for name, labels in code_labels.items():  # go through each coded variable
    label_order = list(labels.values())  # labels in code order, so tables sort the same way
    data[name + "_label"] = pd.Categorical(data[name].map(labels), categories=label_order, ordered=True)  # add the text label column
    unlabeled = (data[name].notna() & data[name + "_label"].isna()).sum()  # codes that found no label
    print(name.ljust(12), "values with no label:", unlabeled)  # should be 0
print("Shape after adding labels (rows, columns):", data.shape)  # loaded columns plus the label columns

## 8. Summary table
One row for every column we loaded. The mean is weighted with `asecwt`. The count (n), the missing count, the min, and the max are unweighted. Numbers show 1 decimal.

For a 0/1 column like `female`, the mean is the share. For a code with several groups like `educ5`, the mean has no meaning by itself. We keep the row for n, missing, min, and max.

In [ ]:
summary_columns = needed_columns  # one row for every column we loaded
summary_rows = []  # list that will hold one row per column
for column in summary_columns:  # go through each column
    has_value = data[column].notna()  # True where the value is not missing
    values = data.loc[has_value, column]  # the values that are not missing
    weights = data.loc[has_value, "asecwt"]  # the weights for those people
    weighted_mean = (values * weights).sum() / weights.sum()  # mean weighted with asecwt
    summary_rows.append({  # add one row to the list
        "column": column,  # name of the column
        "n (unweighted)": has_value.sum(),  # people with a value
        "missing": (~has_value).sum(),  # people without a value
        "weighted mean": weighted_mean,  # average weighted with asecwt
        "min": values.min(),  # smallest value
        "max": values.max(),  # largest value
    })  # end of this row
summary_table = pd.DataFrame(summary_rows).set_index("column")  # turn the list into a table
print("Summary table (mean weighted with asecwt)")  # title above the table
print(summary_table.round(1))  # show the table with 1 decimal